# SW-S4 reusable back-analysis after a simulation

Run this notebook after a simulation writes its CSV into `SWS4/results_csv`, or add an external mounted result directory in the configuration cell. It catalogs all 24 current SWS4 input decks, loads each available result independently, rejects extrapolation of partial histories, scores the eight paper-facing observations, checks numerical integrity, compares parent and candidate variants, and audits the actual input-deck differences.

The default configuration evaluates the current MC and BBFast variant families. Every input and expected CSV filename is visible in `CASE_CATALOG`. For a new study, edit `ANALYSIS_GROUPS`, and optionally `EXTRA_RESULTS_DIRS` or `CASE_FILE_OVERRIDES`. Read the status and gate tables before interpreting fit scores.


In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.max_rows', 300)

T_REF = 55.0
EXPECTED_END_S = 3500.0
SAVE_TABLES = False  # Set True to write CSV summaries after every completed run.
SAVE_FIGURES = False

EXTRA_RESULTS_DIRS = [
    # Path('/mounted/cluster/results'),
]
CASE_FILE_OVERRIDES = {
    # 'case_stem': Path('/exact/path/to/case_stem.csv'),
}

ANALYSIS_GROUPS = {
    'MC 67_11 variants': {
        'parent': '67_11_sw4_mc_dS0p15_s28_w12_m0',
        'candidates': [
            '67_11_sw4_mc_dS0p15_s28_w12_m0_kernel_SV',
            '67_11_sw4_mc_dS0p15_s28_w12_m0_kernel_SV_mesh3',
        ],
    },
    'BBFast 68_01 variants': {
        'parent': '68_01_sw4_bbfast_tail6p50_eta3p50_m0',
        'candidates': [
            '68_01_sw4_bbfast_tail6p50_eta3p50_m0_kernel_SV',
            '68_01_sw4_bbfast_tail6p50_eta3p50_m0_kernel_SV_mesh3',
        ],
    },
    'BBFast 68_02 variants': {
        'parent': '68_02_sw4_bbfast_tail6p75_eta3p25_m0',
        'candidates': [
            '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar',
            '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_cyclic',
            '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_cyclic_kernel_SV',
            '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_cyclic_kernel_SV_mesh3',
            '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_kernel_SV',
            '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_kernel_SV_mesh3',
            '68_02_sw4_bbfast_tail6p75_eta3p25_m0_cyclic',
            '68_02_sw4_bbfast_tail6p75_eta3p25_m0_cyclic_kernel_SV',
            '68_02_sw4_bbfast_tail6p75_eta3p25_m0_cyclic_kernel_SV_mesh3',
            '68_02_sw4_bbfast_tail6p75_eta3p25_m0_kernel_SV',
            '68_02_sw4_bbfast_tail6p75_eta3p25_m0_kernel_SV_mesh3',
            '68_02_sw4_bbfast_tail6p75_eta3p25_m0_shutin',
            '68_02_sw4_bbfast_tail6p75_eta3p25_m0_shutin_kernel_SV',
            '68_02_sw4_bbfast_tail6p75_eta3p25_m0_shutin_kernel_SV_mesh3',
        ],
    },
    'BBFast 68_03 variants': {
        'parent': '68_03_sw4_bbfast_tail6p50_eta3p25_m0',
        'candidates': [
            '68_03_sw4_bbfast_tail6p50_eta3p25_m0_kernel_SV',
            '68_03_sw4_bbfast_tail6p50_eta3p25_m0_kernel_SV_mesh3',
        ],
    },
    'BBFast 89 theta30 HPC comparison': {
        'parent': '89_01_sw4_bbfast_theta30_paperjrc_kernel_SV_biot0p6_hpc',
        'candidates': [
            '89_06_sw4_bbfast_theta30_kernel_SV_biot0p6_hpc',
            '90_07_sw4_bbfast_theta30_jrc9_kernel_SV_biot0p6_hpc',
            '90_08_sw4_bbfast_theta30_jrc5_kernel_SV_biot0p6_hpc',
        ],
    },
}

ENDPOINT_TOLERANCES = {
    'shear_stress_end_MPa': 0.35,
    'shear_slip_end_mm': 0.0015,
    'normal_dilation_min_mm': 0.0020,
    'normal_dilation_end_mm': 0.0020,
}


## 1. Locate the current SWS4 folder and catalog every input deck

`CASE_CATALOG` explicitly lists every analyzed SWS4 result and its matching top-level `.i` file. It scans both `results_csv/` and `results_csv_hpc_rorqual/`. `INPUT_CATALOG` turns those definitions into a table, while `LINEAGE` records the current parent/variant groups.


In [ ]:
SAMPLE = 'SWS4'
cwd = Path.cwd().resolve()
search_roots = [cwd, *cwd.parents]
location_candidates = search_roots + [
    root / 'Examples' / 'YeGhasemmi2018' / SAMPLE for root in search_roots
]
seen = set()
location_candidates = [
    path for path in location_candidates
    if not (str(path) in seen or seen.add(str(path)))
]
SW4_DIR = next(
    (
        path.resolve() for path in location_candidates
        if path.name == SAMPLE
        and (path / SAMPLE).is_dir()
        and (path / 'results_csv').is_dir()
    ),
    None,
)
if SW4_DIR is None:
    searched = '\n'.join(str(path) for path in location_candidates)
    raise FileNotFoundError(
        f'Could not locate the {SAMPLE} folder with {SAMPLE}/ validation data '
        f'and results_csv/. Searched:\n{searched}'
    )

VALIDATION_DIR = SW4_DIR / SAMPLE
RESULTS_DIR = SW4_DIR / 'results_csv'
HPC_RESULTS_DIR = SW4_DIR / 'results_csv_hpc_rorqual'
RESULT_DIRS = [RESULTS_DIR, HPC_RESULTS_DIR]
OUTPUT_DIR = SW4_DIR / 'back_analysis_output'

# Manual analysis selection. Only uncommented CASE_CATALOG entries can be loaded.
# Comment out a complete entry to hide it; filesystem discovery cannot add it back.
CASE_CATALOG = {
    '67_11_sw4_mc_dS0p15_s28_w12_m0': {
        'csv': RESULTS_DIR / '67_11_sw4_mc_dS0p15_s28_w12_m0.csv',
        'input': SW4_DIR / '67_11_sw4_mc_dS0p15_s28_w12_m0.i',
        'style': '-.',
        'color': 'tab:blue',
        'kind': 'simulation',
        'required': True,
    },
    '67_11_sw4_mc_dS0p15_s28_w12_m0_kernel_SV': {
        'csv': RESULTS_DIR / '67_11_sw4_mc_dS0p15_s28_w12_m0_kernel_SV.csv',
        'input': SW4_DIR / '67_11_sw4_mc_dS0p15_s28_w12_m0_kernel_SV.i',
        'style': '-',
        'color': 'tab:red',
        'kind': 'simulation',
        'required': True,
    },
    '67_11_sw4_mc_dS0p15_s28_w12_m0_kernel_SV_mesh3': {
        'csv': RESULTS_DIR / '67_11_sw4_mc_dS0p15_s28_w12_m0_kernel_SV_mesh3.csv',
        'input': SW4_DIR / '67_11_sw4_mc_dS0p15_s28_w12_m0_kernel_SV_mesh3.i',
        'style': '--',
        'color': 'tab:green',
        'kind': 'simulation',
        'required': True,
    },
    '68_01_sw4_bbfast_tail6p50_eta3p50_m0': {
        'csv': RESULTS_DIR / '68_01_sw4_bbfast_tail6p50_eta3p50_m0.csv',
        'input': SW4_DIR / '68_01_sw4_bbfast_tail6p50_eta3p50_m0.i',
        'style': ':',
        'color': 'tab:orange',
        'kind': 'simulation',
        'required': True,
    },
    '68_01_sw4_bbfast_tail6p50_eta3p50_m0_kernel_SV': {
        'csv': RESULTS_DIR / '68_01_sw4_bbfast_tail6p50_eta3p50_m0_kernel_SV.csv',
        'input': SW4_DIR / '68_01_sw4_bbfast_tail6p50_eta3p50_m0_kernel_SV.i',
        'style': '-.',
        'color': 'tab:purple',
        'kind': 'simulation',
        'required': True,
    },
    '68_01_sw4_bbfast_tail6p50_eta3p50_m0_kernel_SV_mesh3': {
        'csv': RESULTS_DIR / '68_01_sw4_bbfast_tail6p50_eta3p50_m0_kernel_SV_mesh3.csv',
        'input': SW4_DIR / '68_01_sw4_bbfast_tail6p50_eta3p50_m0_kernel_SV_mesh3.i',
        'style': '-',
        'color': 'tab:brown',
        'kind': 'simulation',
        'required': True,
    },
    '68_02_sw4_bbfast_tail6p75_eta3p25_m0': {
        'csv': RESULTS_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0.csv',
        'input': SW4_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0.i',
        'style': '--',
        'color': 'tab:pink',
        'kind': 'simulation',
        'required': True,
    },
    '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar': {
        'csv': RESULTS_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar.csv',
        'input': SW4_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar.i',
        'style': ':',
        'color': 'tab:gray',
        'kind': 'simulation',
        'required': True,
    },
    '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_cyclic': {
        'csv': RESULTS_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_cyclic.csv',
        'input': SW4_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_cyclic.i',
        'style': '-.',
        'color': 'tab:olive',
        'kind': 'simulation',
        'required': True,
    },
    '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_cyclic_kernel_SV': {
        'csv': RESULTS_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_cyclic_kernel_SV.csv',
        'input': SW4_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_cyclic_kernel_SV.i',
        'style': '-',
        'color': 'tab:cyan',
        'kind': 'simulation',
        'required': True,
    },
    '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_cyclic_kernel_SV_mesh3': {
        'csv': RESULTS_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_cyclic_kernel_SV_mesh3.csv',
        'input': SW4_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_cyclic_kernel_SV_mesh3.i',
        'style': '--',
        'color': 'tab:blue',
        'kind': 'simulation',
        'required': True,
    },
    '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_kernel_SV': {
        'csv': RESULTS_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_kernel_SV.csv',
        'input': SW4_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_kernel_SV.i',
        'style': ':',
        'color': 'tab:red',
        'kind': 'simulation',
        'required': True,
    },
    '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_kernel_SV_mesh3': {
        'csv': RESULTS_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_kernel_SV_mesh3.csv',
        'input': SW4_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_bakhtar_kernel_SV_mesh3.i',
        'style': '-.',
        'color': 'tab:green',
        'kind': 'simulation',
        'required': True,
    },
    '68_02_sw4_bbfast_tail6p75_eta3p25_m0_cyclic': {
        'csv': RESULTS_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_cyclic.csv',
        'input': SW4_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_cyclic.i',
        'style': '-',
        'color': 'tab:orange',
        'kind': 'simulation',
        'required': True,
    },
    '68_02_sw4_bbfast_tail6p75_eta3p25_m0_cyclic_kernel_SV': {
        'csv': RESULTS_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_cyclic_kernel_SV.csv',
        'input': SW4_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_cyclic_kernel_SV.i',
        'style': '--',
        'color': 'tab:purple',
        'kind': 'simulation',
        'required': True,
    },
    '68_02_sw4_bbfast_tail6p75_eta3p25_m0_cyclic_kernel_SV_mesh3': {
        'csv': RESULTS_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_cyclic_kernel_SV_mesh3.csv',
        'input': SW4_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_cyclic_kernel_SV_mesh3.i',
        'style': ':',
        'color': 'tab:brown',
        'kind': 'simulation',
        'required': True,
    },
    '68_02_sw4_bbfast_tail6p75_eta3p25_m0_kernel_SV': {
        'csv': RESULTS_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_kernel_SV.csv',
        'input': SW4_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_kernel_SV.i',
        'style': '-.',
        'color': 'tab:pink',
        'kind': 'simulation',
        'required': True,
    },
    '68_02_sw4_bbfast_tail6p75_eta3p25_m0_kernel_SV_mesh3': {
        'csv': RESULTS_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_kernel_SV_mesh3.csv',
        'input': SW4_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_kernel_SV_mesh3.i',
        'style': '-',
        'color': 'tab:gray',
        'kind': 'simulation',
        'required': True,
    },
    '68_02_sw4_bbfast_tail6p75_eta3p25_m0_shutin': {
        'csv': RESULTS_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_shutin.csv',
        'input': SW4_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_shutin.i',
        'style': '--',
        'color': 'tab:olive',
        'kind': 'simulation',
        'required': True,
    },
    '68_02_sw4_bbfast_tail6p75_eta3p25_m0_shutin_kernel_SV': {
        'csv': RESULTS_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_shutin_kernel_SV.csv',
        'input': SW4_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_shutin_kernel_SV.i',
        'style': ':',
        'color': 'tab:cyan',
        'kind': 'simulation',
        'required': True,
    },
    '68_02_sw4_bbfast_tail6p75_eta3p25_m0_shutin_kernel_SV_mesh3': {
        'csv': RESULTS_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_shutin_kernel_SV_mesh3.csv',
        'input': SW4_DIR / '68_02_sw4_bbfast_tail6p75_eta3p25_m0_shutin_kernel_SV_mesh3.i',
        'style': '-.',
        'color': 'tab:blue',
        'kind': 'simulation',
        'required': True,
    },
    '68_03_sw4_bbfast_tail6p50_eta3p25_m0': {
        'csv': RESULTS_DIR / '68_03_sw4_bbfast_tail6p50_eta3p25_m0.csv',
        'input': SW4_DIR / '68_03_sw4_bbfast_tail6p50_eta3p25_m0.i',
        'style': '-',
        'color': 'tab:red',
        'kind': 'simulation',
        'required': True,
    },
    '68_03_sw4_bbfast_tail6p50_eta3p25_m0_kernel_SV': {
        'csv': RESULTS_DIR / '68_03_sw4_bbfast_tail6p50_eta3p25_m0_kernel_SV.csv',
        'input': SW4_DIR / '68_03_sw4_bbfast_tail6p50_eta3p25_m0_kernel_SV.i',
        'style': '--',
        'color': 'tab:green',
        'kind': 'simulation',
        'required': True,
    },
    '68_03_sw4_bbfast_tail6p50_eta3p25_m0_kernel_SV_mesh3': {
        'csv': RESULTS_DIR / '68_03_sw4_bbfast_tail6p50_eta3p25_m0_kernel_SV_mesh3.csv',
        'input': SW4_DIR / '68_03_sw4_bbfast_tail6p50_eta3p25_m0_kernel_SV_mesh3.i',
        'style': ':',
        'color': 'tab:orange',
        'kind': 'simulation',
        'required': True,
    },
    # run on Rorqual
    # ran on 32 CPU cores
    '89_01_sw4_bbfast_theta30_paperjrc_kernel_SV_biot0p6_hpc': {
        'csv': HPC_RESULTS_DIR / '89_01_sw4_bbfast_theta30_paperjrc_kernel_SV_biot0p6_hpc.csv',
        'input': SW4_DIR / '89_01_sw4_bbfast_theta30_paperjrc_kernel_SV_biot0p6.i',
        'style': '-.',
        'color': 'tab:olive',
        'kind': 'simulation',
        'required': True,
    },
    # run on Rorqual
    # ran on 32 CPU cores
    '89_06_sw4_bbfast_theta30_kernel_SV_biot0p6_hpc': {
        'csv': HPC_RESULTS_DIR / '89_06_sw4_bbfast_theta30_kernel_SV_biot0p6_hpc.csv',
        'input': SW4_DIR / '89_06_sw4_bbfast_theta30_kernel_SV_biot0p6.i',
        'style': '-',
        'color': 'tab:cyan',
        'kind': 'simulation',
        'required': True,
    },
    # run on Rorqual
    # ran on 32 CPU cores
    '90_07_sw4_bbfast_theta30_jrc9_kernel_SV_biot0p6_hpc': {
        'csv': HPC_RESULTS_DIR / '90_07_sw4_bbfast_theta30_jrc9_kernel_SV_biot0p6_hpc.csv',
        'input': SW4_DIR / '90_07_sw4_bbfast_theta30_jrc9_kernel_SV_biot0p6.i',
        'style': ':',
        'color': 'tab:blue',
        'kind': 'simulation',
        'required': True,
    },
    # run on Rorqual
    # ran on 32 CPU cores
    '90_08_sw4_bbfast_theta30_jrc5_kernel_SV_biot0p6_hpc': {
        'csv': HPC_RESULTS_DIR / '90_08_sw4_bbfast_theta30_jrc5_kernel_SV_biot0p6_hpc.csv',
        'input': SW4_DIR / '90_08_sw4_bbfast_theta30_jrc5_kernel_SV_biot0p6.i',
        'style': '--',
        'color': 'tab:purple',
        'kind': 'simulation',
        'required': True,
    },
}

# Remove commented candidates from each comparison group. Commenting a parent hides its group.
ANALYSIS_GROUPS = {
    group_name: {
        'parent': group['parent'],
        'candidates': [case for case in group['candidates'] if case in CASE_CATALOG],
    }
    for group_name, group in ANALYSIS_GROUPS.items()
    if group['parent'] in CASE_CATALOG
}

input_paths = [cfg['input'] for cfg in CASE_CATALOG.values()]
missing_inputs = [path for path in input_paths if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(
        'Cataloged input decks are missing:\n' + '\n'.join(str(path) for path in missing_inputs)
    )

INPUT_CATALOG = pd.DataFrame([
    {
        'case': case,
        'relative_input_path': str(cfg['input'].relative_to(SW4_DIR)),
        'expected_csv': str(cfg['csv'].relative_to(SW4_DIR)),
        'style': cfg['style'],
        'color': cfg['color'],
        'kind': cfg['kind'],
    }
    for case, cfg in CASE_CATALOG.items()
])
print(f'SW4 directory: {SW4_DIR}')
print(f'Cataloged {len(INPUT_CATALOG)} current input decks.')
display(INPUT_CATALOG)


In [ ]:
lineage_rows = []
for group_name, group in ANALYSIS_GROUPS.items():
    parent = group['parent']
    for role, case in [('parent', parent), *[('candidate', stem) for stem in group['candidates']]]:
        cfg = CASE_CATALOG[case]
        lineage_rows.append({
            'case': case,
            'law': 'MC' if case.startswith('67_11_') else 'BBFast',
            'stage': group_name,
            'role': role,
            'parent': '' if role == 'parent' else parent,
            'input_path': cfg['input'],
            'input_exists': cfg['input'].is_file(),
            'relative_input_path': str(cfg['input'].relative_to(SW4_DIR)),
            'expected_csv': str(cfg['csv'].relative_to(SW4_DIR)),
        })

LINEAGE = pd.DataFrame(lineage_rows).drop_duplicates('case', keep='last')
display(LINEAGE[['case', 'law', 'stage', 'role', 'parent', 'input_exists', 'relative_input_path', 'expected_csv']])


## 2. Discover and load requested outputs independently

Selection prefers the readable exact-name CSV with the greatest final time, then the greatest number of rows, then the newest modification time. An explicit override always wins. Every selected path and every ambiguity is shown.


In [ ]:
def unique_in_order(values):
    return list(dict.fromkeys(values))

FOCUS_CASES = unique_in_order([
    stem
    for group in ANALYSIS_GROUPS.values()
    for stem in [group['parent'], *group['candidates']]
])

csv_paths = [
    path
    for result_dir in RESULT_DIRS
    if result_dir.is_dir()
    for path in result_dir.glob('*.csv')
]
for extra in EXTRA_RESULTS_DIRS:
    extra = Path(extra).expanduser().resolve()
    if extra.is_dir():
        csv_paths.extend(extra.rglob('*.csv'))

CSV_INDEX = {stem: [] for stem in FOCUS_CASES}
for path in csv_paths:
    matches = [stem for stem in FOCUS_CASES if path.stem.startswith(stem)]
    if matches:
        CSV_INDEX[max(matches, key=len)].append(path.resolve())

def read_result(path):
    data = pd.read_csv(path, on_bad_lines='skip')
    data.columns = [str(column).strip() for column in data.columns]
    if 'time' not in data:
        raise ValueError('missing time column')
    data['time'] = pd.to_numeric(data['time'], errors='coerce')
    data = data.dropna(subset=['time']).sort_values('time')
    data = data.drop_duplicates('time', keep='last').reset_index(drop=True)
    if data.empty:
        raise ValueError('no readable time rows')
    return data

def input_candidates(stem):
    cfg = CASE_CATALOG.get(stem)
    return [cfg['input']] if cfg and cfg['input'].is_file() else []

CASES = {}
SELECTED_INPUTS = {}
status_rows = []
candidate_paths = {}
for stem in FOCUS_CASES:
    override = CASE_FILE_OVERRIDES.get(stem)
    paths = [Path(override).expanduser().resolve()] if override else CSV_INDEX.get(stem, [])
    readable = []
    errors = []
    for path in unique_in_order(paths):
        try:
            data = read_result(path)
            score = (float(data['time'].iloc[-1]), len(data), path.stat().st_mtime)
            readable.append((score, path, data))
        except Exception as exc:
            errors.append(f'{path}: {exc}')
    if readable:
        readable.sort(key=lambda item: item[0], reverse=True)
        _, selected_csv, data = readable[0]
        CASES[stem] = data
        last_time = float(data['time'].iloc[-1])
        intended_end = 5000.0 if '_shutin' in stem else EXPECTED_END_S
        state = 'complete' if last_time >= intended_end - 1.0 else 'partial'
        provenance = selected_csv.with_suffix('.provenance.txt').is_file()
        candidate_paths[stem] = [str(item[1]) for item in readable]
    else:
        selected_csv = None
        last_time = np.nan
        state = 'missing'
        provenance = False
        candidate_paths[stem] = errors
    decks = input_candidates(stem)
    selected_input = decks[0] if len(decks) == 1 else None
    SELECTED_INPUTS[stem] = selected_input
    status_rows.append({
        'case': stem, 'status': state, 'last_time_s': last_time,
        'rows': len(CASES.get(stem, [])), 'provenance': provenance,
        'selected_csv': str(selected_csv) if selected_csv else '',
        'input_match_count': len(decks),
        'selected_input': str(selected_input) if selected_input else '',
        'csv_match_count': len(readable),
    })
STATUS = pd.DataFrame(status_rows)
display(STATUS.style.format({'last_time_s': '{:.1f}'}))
ambiguous_inputs = STATUS[STATUS['input_match_count'].ne(1)]
if not ambiguous_inputs.empty:
    print('Resolve ambiguous or missing input decks before using parameter differences:')
    display(ambiguous_inputs)
duplicate_results = STATUS[STATUS['csv_match_count'].gt(1)]
if not duplicate_results.empty:
    print('Multiple readable result copies were found. The selected longest/newest copy is shown in STATUS; use CASE_FILE_OVERRIDES when that choice is not intended.')
    display(duplicate_results[['case', 'selected_csv', 'csv_match_count']])
    for stem in duplicate_results['case']:
        print(stem, candidate_paths.get(stem, []))


## 3. Validation definitions and full-history scores

These definitions match the corrected audit notebook. Mechanical curves begin at the 55 s reference. A partial history is scored only at digitized times that it reached. No curve is extrapolated.


In [ ]:
VALIDATION_FILES = {
    'injection_pressure': 'Ye2018_SW4_Injection_pressure_Vs_time.csv',
    'differential_stress': 'Ye2018_SW4_Differential_Stress_Vs_time.csv',
    'effective_normal': 'Ye2018_SW4_normal_stress_Vs_time.csv',
    'shear_stress': 'Ye2018_SW4_shear_stress_Vs_time.csv',
    'shear_slip': 'Ye2018_SW4_shear_slip_Vs_time.csv',
    'normal_dilation': 'Ye2018_SW4_normal_dilation_Vs_time.csv',
    'permeability': 'Ye2018_SW4_frac_perm_Vs_time.csv',
    'flow_rate': 'Ye2018_SW4_flow_rate_Vs_time.csv',
}
OBS = {
    'injection_pressure': {'column': 'injection_pressure_pp', 'scale': 1e-6, 'reference': False, 'mechanical': False, 'label': 'Injection pressure (MPa)'},
    'differential_stress': {'column': 'differential_stress_reaction_mpa_pp', 'scale': 1.0, 'reference': False, 'mechanical': True, 'label': 'Reaction differential stress (MPa)'},
    'effective_normal': {'column': 'effective_normal_compression_mpa_pp', 'scale': 1.0, 'reference': False, 'mechanical': True, 'label': 'Effective normal compression (MPa)'},
    'shear_stress': {'column': 'shear_traction_magnitude_pa', 'scale': 1e-6, 'reference': False, 'mechanical': True, 'label': 'Shear stress (MPa)'},
    'shear_slip': {'column': 'czm_shear_slip_mm_pp', 'scale': 1.0, 'reference': True, 'mechanical': True, 'label': 'Local CZM shear slip from 55 s (mm)'},
    'normal_dilation': {'column': 'czm_normal_dilation_paper_mm_pp', 'scale': 1.0, 'reference': True, 'mechanical': True, 'label': 'Local CZM normal dilation from 55 s (mm)'},
    'permeability': {'column': 'fracture_permeability_pp', 'scale': 1.0, 'reference': False, 'mechanical': False, 'label': 'Fracture permeability (m²)'},
    'flow_rate': {'column': 'flow_rate_validation_ml_min_pp', 'scale': 1.0, 'reference': False, 'mechanical': False, 'label': 'Eq. (9) derived flow rate (mL/min)'},
}
MECHANICAL_SCORE_KEYS = ['differential_stress', 'shear_stress', 'shear_slip', 'normal_dilation']

VALIDATION = {}
for key, filename in VALIDATION_FILES.items():
    values = pd.read_csv(VALIDATION_DIR / filename, header=None, names=['time', 'value'])
    values = values.apply(pd.to_numeric, errors='coerce').dropna().sort_values('time')
    if OBS[key]['reference']:
        values['value'] -= np.interp(T_REF, values['time'], values['value'])
    VALIDATION[key] = values.reset_index(drop=True)

def numeric_column(data, column):
    if column not in data:
        return None
    return pd.to_numeric(data[column], errors='coerce').to_numpy(dtype=float)

def model_series(data, key):
    spec = OBS[key]
    time_all = pd.to_numeric(data['time'], errors='coerce').to_numpy(dtype=float)
    values_all = numeric_column(data, spec['column'])
    if values_all is None:
        return None, None
    values_all = values_all * spec['scale']
    finite_all = np.isfinite(time_all) & np.isfinite(values_all)
    mask = finite_all & (time_all > 0.0)
    if spec['mechanical']:
        mask &= time_all >= T_REF
    time, values = time_all[mask], values_all[mask]
    if spec['reference']:
        if np.any(finite_all) and time_all[finite_all][0] <= T_REF <= time_all[finite_all][-1]:
            values = values - np.interp(T_REF, time_all[finite_all], values_all[finite_all])
        else:
            values = np.full_like(values, np.nan)
    return time, values

def validation_score(data, key):
    validation = VALIDATION[key]
    vt = validation['time'].to_numpy(dtype=float)
    vy = validation['value'].to_numpy(dtype=float)
    mt, my = model_series(data, key)
    empty = {'n': 0, 'coverage': 0.0, 'rmse': np.nan, 'nrmse': np.nan}
    if mt is None or len(mt) < 2:
        return empty
    inside = (vt >= mt[0]) & (vt <= mt[-1])
    if not np.any(inside):
        return empty
    predicted = np.interp(vt[inside], mt, my)
    valid = np.isfinite(predicted) & np.isfinite(vy[inside])
    residual = predicted[valid] - vy[inside][valid]
    rmse = float(np.sqrt(np.mean(residual**2))) if residual.size else np.nan
    span = float(np.ptp(vy))
    return {
        'n': int(valid.sum()), 'coverage': float(valid.sum() / len(vt)),
        'rmse': rmse, 'nrmse': rmse / span if np.isfinite(rmse) and span > 0 else np.nan,
    }


In [ ]:
def endpoint(data, key, mode='end'):
    _, values = model_series(data, key)
    if values is None or not np.any(np.isfinite(values)):
        return np.nan
    finite = values[np.isfinite(values)]
    return float(np.min(finite) if mode == 'min' else finite[-1])

TARGETS = {
    'shear_stress_end_MPa': float(VALIDATION['shear_stress']['value'].iloc[-1]),
    'shear_slip_end_mm': float(VALIDATION['shear_slip']['value'].iloc[-1]),
    'normal_dilation_min_mm': float(VALIDATION['normal_dilation']['value'].min()),
    'normal_dilation_end_mm': float(VALIDATION['normal_dilation']['value'].iloc[-1]),
}
metric_rows = []
summary_rows = []
for group_name, group in ANALYSIS_GROUPS.items():
    for role, stem in [('parent', group['parent']), *[('candidate', s) for s in group['candidates']]]:
        data = CASES.get(stem)
        if data is None:
            summary_rows.append({'group': group_name, 'role': role, 'case': stem, 'status': 'missing'})
            continue
        scores = {key: validation_score(data, key) for key in OBS}
        for key, score in scores.items():
            metric_rows.append({'group': group_name, 'role': role, 'case': stem, 'observable': key, **score})
        mechanical_mean = np.nanmean([scores[key]['nrmse'] for key in MECHANICAL_SCORE_KEYS])
        all_mean = np.nanmean([scores[key]['nrmse'] for key in OBS])
        row = {
            'group': group_name, 'role': role, 'case': stem,
            'status': STATUS.loc[STATUS['case'].eq(stem), 'status'].iloc[0],
            'mechanical_mean_nrmse': mechanical_mean,
            'all_observable_mean_nrmse': all_mean,
            'shear_stress_end_MPa': endpoint(data, 'shear_stress'),
            'shear_slip_end_mm': endpoint(data, 'shear_slip'),
            'normal_dilation_min_mm': endpoint(data, 'normal_dilation', 'min'),
            'normal_dilation_end_mm': endpoint(data, 'normal_dilation'),
        }
        for target_name, target in TARGETS.items():
            row[target_name + '_error'] = row[target_name] - target
        row['endpoint_gate_pass'] = bool(
            row['status'] == 'complete'
            and all(abs(row[name + '_error']) <= tolerance for name, tolerance in ENDPOINT_TOLERANCES.items())
        )
        summary_rows.append(row)
METRICS = pd.DataFrame(metric_rows)
SCORECARD = pd.DataFrame(summary_rows)
display(pd.DataFrame([{'target': key, 'value': value, 'tolerance': ENDPOINT_TOLERANCES[key]} for key, value in TARGETS.items()]))
display(SCORECARD.style.format({
    'mechanical_mean_nrmse': '{:.5f}', 'all_observable_mean_nrmse': '{:.5f}',
    'shear_stress_end_MPa': '{:.3f}', 'shear_slip_end_mm': '{:.5f}',
    'normal_dilation_min_mm': '{:.5f}', 'normal_dilation_end_mm': '{:.5f}',
    **{column: '{:+.5f}' for column in SCORECARD if column.endswith('_error')},
}))
display(METRICS.style.format({'coverage': '{:.0%}', 'rmse': '{:.5g}', 'nrmse': '{:.4f}'}))


## 4. Plot complete histories and residual shapes

A sign-changing residual usually cannot be fixed with one uniform strength shift. The residual plots are as important as the overlay.


In [ ]:
GROUP_COLORS = ['0.35', 'tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']
for group_name, group in ANALYSIS_GROUPS.items():
    stems = [group['parent'], *group['candidates']]
    fig, axes = plt.subplots(4, 2, figsize=(15, 15), sharex=True)
    for ax, key in zip(axes.flat, OBS):
        validation = VALIDATION[key]
        ax.scatter(validation['time'], validation['value'], s=14, c='black', label='Ye et al.', zorder=5)
        for index, stem in enumerate(stems):
            data = CASES.get(stem)
            if data is None:
                continue
            time, values = model_series(data, key)
            if time is None or not len(time):
                continue
            label = ('parent ' if index == 0 else '') + stem.split('_sw4_', 1)[0]
            color = GROUP_COLORS[index % len(GROUP_COLORS)]
            ax.plot(time, values, color=color, ls=':' if index == 0 else '-', lw=1.8, label=label)
        ax.set_title(key.replace('_', ' ').title())
        ax.set_ylabel(OBS[key]['label'])
        ax.legend(fontsize=7)
    for ax in axes[-1]:
        ax.set_xlabel('Time (s)')
    fig.suptitle(group_name + ' — validation histories', y=1.005)
    fig.tight_layout()
    if SAVE_FIGURES:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUTPUT_DIR / (re.sub(r'[^A-Za-z0-9_.-]+', '_', group_name) + '_histories.png'), dpi=200, bbox_inches='tight')
    plt.show()

    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    for ax, key in zip(axes.flat, MECHANICAL_SCORE_KEYS):
        validation = VALIDATION[key]
        vt = validation['time'].to_numpy(dtype=float)
        vy = validation['value'].to_numpy(dtype=float)
        for index, stem in enumerate(stems):
            data = CASES.get(stem)
            if data is None:
                continue
            mt, my = model_series(data, key)
            inside = (vt >= mt[0]) & (vt <= mt[-1]) if mt is not None and len(mt) else np.zeros(len(vt), dtype=bool)
            if np.any(inside):
                residual = np.interp(vt[inside], mt, my) - vy[inside]
                color = GROUP_COLORS[index % len(GROUP_COLORS)]
                ax.plot(vt[inside], residual, color=color, ls=':' if index == 0 else '-', label=stem.split('_sw4_', 1)[0])
        ax.axhline(0.0, color='black', lw=0.8)
        ax.set_title(key.replace('_', ' ').title() + ' residual')
        ax.set_ylabel(OBS[key]['label'])
        ax.legend(fontsize=7)
    for ax in axes[-1]:
        ax.set_xlabel('Time (s)')
    fig.suptitle(group_name + ' — model minus experiment', y=1.01)
    fig.tight_layout()
    plt.show()


## 5. Numerical-integrity and evidence gates

Mechanical calibration integrity and independent solved-flow integrity are separated deliberately. A large mass-imbalance diagnostic blocks a solved-flow validation claim but does not erase the value of a completed case for diagnosing mechanical parameter response. Eq. (9) flow remains a derived comparison.


In [ ]:
EXPECTED_FAULT_AREA = 2.0 * np.pi * 0.025255**2
gate_rows = []
for stem in FOCUS_CASES:
    data = CASES.get(stem)
    status_row = STATUS[STATUS['case'].eq(stem)].iloc[0]
    row = {
        'case': stem, 'status': status_row['status'],
        'provenance_recorded': bool(status_row['provenance']),
    }
    if data is None:
        gate_rows.append(row)
        continue
    time = pd.to_numeric(data['time'], errors='coerce').to_numpy(dtype=float)
    after_ref = np.isfinite(time) & (time >= T_REF)
    area = numeric_column(data, 'fracture_interface_area_pp')
    mass = numeric_column(data, 'flow_mass_imbalance_fraction_pp')
    spring_gap = numeric_column(data, 'reaction_vs_machine_spring_mpa_pp')
    row['fault_area_rel_error'] = abs(np.nanmedian(area) - EXPECTED_FAULT_AREA) / EXPECTED_FAULT_AREA if area is not None else np.nan
    mass_after = np.abs(mass[after_ref]) if mass is not None else np.array([])
    row['mass_imbalance_max_after_ref'] = float(np.nanmax(mass_after)) if mass_after.size else np.nan
    row['mass_imbalance_p95_after_ref'] = float(np.nanquantile(mass_after, 0.95)) if mass_after.size else np.nan
    row['mass_imbalance_end'] = float(mass_after[np.flatnonzero(np.isfinite(mass_after))[-1]]) if np.any(np.isfinite(mass_after)) else np.nan
    row['max_abs_reaction_spring_gap_MPa'] = float(np.nanmax(np.abs(spring_gap[after_ref]))) if spring_gap is not None and np.any(after_ref) else np.nan
    row['area_gate_pass'] = bool(np.isfinite(row['fault_area_rel_error']) and row['fault_area_rel_error'] <= 0.05)
    row['reaction_gate_pass'] = bool(np.isfinite(row['max_abs_reaction_spring_gap_MPa']) and row['max_abs_reaction_spring_gap_MPa'] <= 0.5)
    row['solved_flow_mass_gate_pass'] = bool(np.isfinite(row['mass_imbalance_max_after_ref']) and row['mass_imbalance_max_after_ref'] <= 0.05)
    row['mechanical_calibration_integrity_pass'] = bool(
        row['status'] == 'complete' and row['provenance_recorded']
        and row['area_gate_pass'] and row['reaction_gate_pass']
    )
    row['independent_solved_flow_integrity_pass'] = bool(
        row['mechanical_calibration_integrity_pass'] and row['solved_flow_mass_gate_pass']
    )
    gate_rows.append(row)
GATES = pd.DataFrame(gate_rows)
display(GATES.style.format({
    'fault_area_rel_error': '{:.2%}',
    'mass_imbalance_max_after_ref': '{:.3g}',
    'mass_imbalance_p95_after_ref': '{:.3g}',
    'mass_imbalance_end': '{:.3g}',
    'max_abs_reaction_spring_gap_MPa': '{:.3g}',
}))


## 6. Audit what actually changed in each input file

Filenames and `cases.csv` describe intent; this section checks the decks themselves. Scalar assignments are indexed by block path and key. Multiline functions are not reduced to a single scalar, so use the displayed input paths and a normal text diff when a load history is the intended change.


In [ ]:
def parse_deck_assignments(path):
    path = Path(path)
    stack = []
    records = {}
    duplicate_counter = {}
    for line_number, raw in enumerate(path.read_text(errors='replace').splitlines(), start=1):
        line = raw.split('#', 1)[0].strip()
        if not line:
            continue
        block_match = re.fullmatch(r'\[([^]]*)\]', line)
        if block_match:
            label = block_match.group(1).strip()
            if label == '':
                stack = []
            elif label.startswith('../'):
                parts = label.split('/')
                pops = sum(part == '..' for part in parts)
                stack = stack[:-pops] if pops <= len(stack) else []
                tail = [part for part in parts if part not in ('', '..', '.') ]
                stack.extend(tail)
            elif label.startswith('./'):
                stack.append(label[2:])
            else:
                stack = [label]
            continue
        assignment = re.match(r'^([A-Za-z_][A-Za-z0-9_]*)\s*=\s*(.+?)\s*$', line)
        if not assignment:
            continue
        key, value = assignment.groups()
        base_path = '/'.join([*stack, key])
        duplicate_counter[base_path] = duplicate_counter.get(base_path, 0) + 1
        unique_path = base_path if duplicate_counter[base_path] == 1 else f'{base_path}#{duplicate_counter[base_path]}'
        records[unique_path] = {'value': value.strip(), 'line': line_number}
    return records

def parameter_category(path):
    lower = path.lower()
    if any(token in lower for token in ['mesh', 'file_mesh']):
        return 'mesh/geometry'
    if any(token in lower for token in ['bc', 'function', 'pressure', 'confin', 'axial', 'piston']):
        return 'boundary/load/HM'
    if any(token in lower for token in ['material', 'friction', 'dilation', 'roughness', 'closure', 'viscosity', 'jrc', 'cohesion']):
        return 'constitutive/hydraulic material'
    if any(token in lower for token in ['executioner', 'solver', 'dt', 'petsc']):
        return 'numerical'
    if any(token in lower for token in ['output', 'file_base', 'postprocessor']):
        return 'output/observation'
    return 'other/global'

def deck_parameter_diff(parent_stem, candidate_stem):
    parent_path = SELECTED_INPUTS.get(parent_stem)
    candidate_path = SELECTED_INPUTS.get(candidate_stem)
    if parent_path is None or candidate_path is None:
        return pd.DataFrame([{'parent': parent_stem, 'candidate': candidate_stem, 'parameter_path': 'INPUT PATH AMBIGUOUS OR MISSING'}])
    parent = parse_deck_assignments(parent_path)
    candidate = parse_deck_assignments(candidate_path)
    rows = []
    for key in sorted(set(parent) | set(candidate)):
        old = parent.get(key, {})
        new = candidate.get(key, {})
        if old.get('value') == new.get('value'):
            continue
        rows.append({
            'parent': parent_stem, 'candidate': candidate_stem,
            'category': parameter_category(key), 'parameter_path': key,
            'parent_value': old.get('value', '<missing>'),
            'candidate_value': new.get('value', '<missing>'),
            'parent_line': old.get('line', np.nan), 'candidate_line': new.get('line', np.nan),
        })
    return pd.DataFrame(rows)

diff_frames = []
for group_name, group in ANALYSIS_GROUPS.items():
    for candidate in group['candidates']:
        frame = deck_parameter_diff(group['parent'], candidate)
        frame.insert(0, 'group', group_name)
        diff_frames.append(frame)
PARAMETER_CHANGES = pd.concat(diff_frames, ignore_index=True, sort=False) if diff_frames else pd.DataFrame()
display(PARAMETER_CHANGES)
print('Review output/file-base changes separately from physical parameters. A deck with unintended BC, mesh, or numerical changes is not an attributable sensitivity case.')


## 7. Parent-relative response and provisional decision

A candidate must improve both the mechanical and all-observable mean NRMSE, pass the endpoint gates, and pass mechanical calibration integrity to be a provisional calibration choice. This deliberately does not convert the solved-flow gate into a mechanical rejection; that gate is reported separately and controls the strength of the hydraulic claim. Scientific judgment and the residual plots still take precedence over this compact rule.


In [ ]:
decision_rows = []
for group_name, group in ANALYSIS_GROUPS.items():
    parent_rows = SCORECARD[
        SCORECARD['case'].eq(group['parent'])
        & SCORECARD['status'].ne('missing')
    ]
    if parent_rows.empty:
        print(f'{group_name}: parent result is unavailable; comparison deferred.')
        continue
    parent = parent_rows.iloc[0]
    for stem in group['candidates']:
        rows = SCORECARD[
            SCORECARD['case'].eq(stem)
            & SCORECARD['status'].ne('missing')
        ]
        if rows.empty:
            continue
        candidate = rows.iloc[0]
        gates = GATES[GATES['case'].eq(stem)]
        mechanical_integrity = bool(gates.iloc[0].get('mechanical_calibration_integrity_pass', False)) if not gates.empty else False
        solved_flow_integrity = bool(gates.iloc[0].get('independent_solved_flow_integrity_pass', False)) if not gates.empty else False
        joint_improvement = bool(
            candidate['mechanical_mean_nrmse'] < parent['mechanical_mean_nrmse']
            and candidate['all_observable_mean_nrmse'] < parent['all_observable_mean_nrmse']
        )
        provisional = bool(candidate['endpoint_gate_pass'] and mechanical_integrity and joint_improvement)
        decision_rows.append({
            'group': group_name, 'case': stem,
            'delta_mechanical_mean_nrmse': candidate['mechanical_mean_nrmse'] - parent['mechanical_mean_nrmse'],
            'delta_all_observable_mean_nrmse': candidate['all_observable_mean_nrmse'] - parent['all_observable_mean_nrmse'],
            'delta_final_shear_stress_MPa': candidate['shear_stress_end_MPa'] - parent['shear_stress_end_MPa'],
            'delta_final_shear_slip_mm': candidate['shear_slip_end_mm'] - parent['shear_slip_end_mm'],
            'delta_dilation_min_mm': candidate['normal_dilation_min_mm'] - parent['normal_dilation_min_mm'],
            'delta_dilation_end_mm': candidate['normal_dilation_end_mm'] - parent['normal_dilation_end_mm'],
            'endpoint_gate_pass': bool(candidate['endpoint_gate_pass']),
            'mechanical_integrity_pass': mechanical_integrity,
            'joint_history_improvement': joint_improvement,
            'provisional_calibration_candidate': provisional,
            'independent_solved_flow_integrity_pass': solved_flow_integrity,
        })

DECISIONS = pd.DataFrame(decision_rows)
if DECISIONS.empty:
    print('No parent/candidate pair currently has enough result data for a promotion decision.')
else:
    display(DECISIONS.style.format({column: '{:+.5f}' for column in DECISIONS if column.startswith('delta_')}))
    for group_name, group in ANALYSIS_GROUPS.items():
        eligible = DECISIONS[
            DECISIONS['group'].eq(group_name) & DECISIONS['provisional_calibration_candidate']
        ].sort_values(['delta_mechanical_mean_nrmse', 'delta_all_observable_mean_nrmse'])
        if eligible.empty:
            print(f'{group_name}: retain parent {group["parent"]}; no candidate passes the compact joint-promotion rule.')
        else:
            best = eligible.iloc[0]
            flow_note = 'with independent solved-flow integrity' if best['independent_solved_flow_integrity_pass'] else 'for mechanical/derived-flow calibration only; solved-flow integrity is not established'
            print(f'{group_name}: provisional choice {best["case"]} ({flow_note}).')


## 8. Optional export and interpretation checklist

Set `SAVE_TABLES = True` in the configuration cell to write the current status, score, gate, parameter-change, and decision tables to `SWS4/back_analysis_output`.


In [ ]:
if SAVE_TABLES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    STATUS.to_csv(OUTPUT_DIR / 'case_status.csv', index=False)
    METRICS.to_csv(OUTPUT_DIR / 'validation_metrics_long.csv', index=False)
    SCORECARD.to_csv(OUTPUT_DIR / 'case_scorecard.csv', index=False)
    GATES.to_csv(OUTPUT_DIR / 'numerical_integrity_gates.csv', index=False)
    PARAMETER_CHANGES.to_csv(OUTPUT_DIR / 'input_parameter_changes.csv', index=False)
    DECISIONS.to_csv(OUTPUT_DIR / 'parent_relative_decisions.csv', index=False)
    INPUT_CATALOG.to_csv(OUTPUT_DIR / 'all_sw4_input_decks.csv', index=False)
    LINEAGE.to_csv(OUTPUT_DIR / 'historical_to_case68_lineage.csv', index=False)
    print(f'Wrote tables to {OUTPUT_DIR}')
else:
    print('Tables were not written. Set SAVE_TABLES = True when persistent output is wanted.')


### Interpretation checklist

1. Never extrapolate a partial history or rank it by an unreached final state.
2. Verify the exact input, result, mesh, executable provenance, and observation columns.
3. Diagnose the residual by preload, onset, rapid event, arrest, and unload intervals.
4. Change a parameter only when its active equation controls the residual in that regime.
5. Keep boundary/load, constitutive, hydraulic, observation, mesh, and numerical changes in separate tests.
6. Use reaction differential stress for the far-field comparison and local CZM displacement as an explicitly limited fracture comparator.
7. Treat Eq. (9) flow as dependent on aperture and pressure; inspect residual flux and mass imbalance separately.
8. Compare final shear stress and final slip together because the compliant machine load line couples them.
9. Do not claim convergence from M0/M1 alone.
10. A parameter set fitted to SW-S4 is calibration. Validation requires held-out evidence.
